In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



# Import CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

# Hardcoded account credentials
username = "aacuser"
password = "thystrup1"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Define the dashboard layout
app.layout = html.Div([
    # Header
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    # Unique identifier
    html.Div("ThomasTET", id="unique-id", style={'fontWeight': 'bold', 'color': 'red', 'textAlign': 'right'}),
    # Logo
    html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
             style={'width': '200px', 'display': 'block', 'margin': 'auto'}),
    html.Hr(),
    html.Div([
        
        # Interactive Rescue Type filter
        html.Label("Rescue Type Filter"),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'WATER'},
                {'label': 'Mountain/Wilderness Rescue', 'value': 'MOUNTAIN'},
                {'label': 'Disaster/Individual Tracking', 'value': 'DISASTER'},
                {'label': 'Reset', 'value': 'RESET'}
            ],
            value='RESET',
            inline=True
        )    
    ]),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
                            # Features for interactive data table to make it user-friendly for client
                            row_selectable="single",
                            selected_rows=[0],
                            page_size=10,
                            sort_action="native",
                            filter_action="native",
                            style_table={'overflowX': 'auto'}
                        ),
    html.Br(),
    html.Hr(),
# This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'selected_rows'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
    """
    Filter DataTable based on Rescue Type selection.
    """
    
    # Define MongoDB queries for each rescue type
    if filter_type == 'WATER':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Labrador Retriever Mix|Chesapeake Bay Retriever|Newfoundland",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == 'MOUNTAIN':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    elif filter_type == 'DISASTER':
        query = {
            "animal_type": "Dog",
            "breed": {
                "$regex": "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler",
                "$options": "i"
            },
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    else:  # RESET
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))
    dff.drop(columns=['_id'], inplace=True, errors='ignore')

    # Always reset selection to first row
    return dff.to_dict('records'), [0]


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    """
    Generate a Pie Chart showing breed distribution for filtered data.
    """
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty or 'breed' not in dff.columns:
        return []

    # Count breeds
    breed_counts = dff['breed'].value_counts()

    # If more than 13 breeds then group remainder as "Other"
    if len(breed_counts) > 13:
        top13 = breed_counts.nlargest(13)
        other_count = breed_counts.iloc[13:].sum()

        # Add "Other" category
        top13["Other"] = other_count
        final_counts = top13
    else:
        final_counts = breed_counts

    fig = px.pie(
        names=final_counts.index,
        values=final_counts.values,
        title='Breed Distribution'
    )

    return [dcc.Graph(figure=fig)]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    """
    Highlight selected columns in the DataTable for visual clarity.
    """
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    """
    Display a map centered on Austin, TX with a marker for the selected dog.
    Tooltip shows breed; popup shows name.
    """
    if viewData is None:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return []

    # Determine selected row index
    if index is None or len(index) == 0 or index[0] >= len(dff):
        row = 0
    else:
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    # Return Leaflet map with marker
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server(mode="inline") 

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
TypeError: 'NoneType' object is not iterable

